# **Telematics Driver Risk Segmentation**

A full pipeline for behavioural driver segmentation from raw OBD telematics data, built for the 2026 McMaster × Co-operators Case Competition.

Data provided and anonymized by the Co-operators Data Science team.

**Pipeline:**
- Clean and validate raw vehicle sensor data across 105 drivers, resolving sampling irregularities and idle trip contamination.
- Engineer behavioural features capturing harsh braking, acceleration, and cornering rates — all normalized by distance to remove exposure bias.
- Segment drivers into risk profiles using K-Medoids clustering; validate against K-Means using silhouette scores.
- Incorporate the risk classification into a Gamma GLM (log link) to generate loss cost predictions on held-out policy data.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
# Directory structure:
# project-root/
#   data/
#     Telematics.csv
#     LC_train.csv
#     LC_test.csv
#   telematics_driver_risk_segmentation.ipynb

tele_data = pd.read_csv("../data/Telematics.csv")

## **Data Structure Exploration**

In [ ]:
# Feature names and data types.
tele_data.info(verbose = True) 

**Observation:** Most features look perfectly normal, but the timestamp feature is of "str" type not "datetime".          <br>
**Interpretation:** The timestamp feature will be useless to us in this state as we cannot perform computations on strings.<br>
**Implication:** We will have to convert this column to "datetime" type in order to check the sampling frequency.        <br>

In [ ]:
# Five-number summaries for all of the numerical features in the dataset
tele_data.describe(include = "all").T

**Observation:** There seems to two different types of features, those which are directly related to the driver (eg. speed), and those which are indirectly related to the driver's actions (eg. coolant_temp_c).

There are many issues we can see from these five number-summaries, namely:

* gps_speed_kph has a maximum value of 512 kph.
* battery has a 75th percentile reading of 0.
* coolant_temp_c and intake_air_temp both have minimum values of -40 degrees celsius.
* imap has a maximum of 236kPa (Normal values hover around 100kPa).
* dtc is always 0.
* "Unnamed: 0" is just an is just a cumulative row counter. <br>

**Interpretation:** Many of these issues include data that is physically impossible, so there are clearly issues in the data collection process. Furthermore, we'll have to explore the relationship between gps_speed_kph and speed, as their distributions are not the same. <br>
**Implication:** We will have to explore these features in order to understand and clean them. Finally, we know that the dtc and "Unnamed: 0" features carry no information, so we should drop them before segmentation. <br>

In [ ]:
# Checking which features have missing values.
tele_data.isna().mean().sort_values(ascending = False)

**Observation:** The magnetometer and accelerometer features have more than zero entries with missing values.          <br>
**Interpretation:** There are some entries where the magnetometer/accelerometer have missing values. Are these because of a handful of trips with sensors that weren't working, or are these scattered across trips?      <br>
**Implication:** We will have to use boolean masking to explore which rows have missing values for these features.          <br>

In [ ]:
# Checking the shape of the dataset.
tele_data.shape

In [ ]:
# Checking how many drivers there are in the dataset.
tele_data['PolicyID'].nunique()

**Final Thoughts:** In terms of features, we have our work cut out for us, as we have to explore 6 of them further, whilst we will have to drop dtc and "Unnamed: 0". We see that there are 105 drivers in the dataset. Additionally, we will have to explore the timestamp feature in order to understand how often samples are being taken.

In [14]:
# Dropping the dtc and "Unnamed: 0" features from our dataset.
tele_data = tele_data.drop(columns = ["dtc", "Unnamed: 0"])

## **Sampling Exploration** 

In this section, we explore the timestamp feature in order to understand it's distribution and how often samples are taken, this will be instrumental in the process of engineering aggregate trip- and driver-level features.

In [ ]:
# Checking the format of the timestamp feature.
tele_data["timestamp"].head()

In [ ]:
# Making a copy of the original dataset.
sampling_data = tele_data.copy()

# Changing data type of timestamp variable to a datetime object.
sampling_data["timestamp"] = pd.to_datetime(sampling_data["timestamp"], format="%d-%m-%Y %H:%M:%S")

# Sort to ensure diffs are calculated in order
sampling_data = sampling_data.sort_values(["PolicyID", "tripID", "timestamp"])

# Calculate avg time diff per trip
trip_sampling = sampling_data.groupby(["PolicyID", "tripID"])["timestamp"].apply(
    lambda x: x.diff().dt.total_seconds().mean() # Take the diff of the column, in datetime seconds, and then compute the mean across the column
).reset_index()
trip_sampling.columns = ["PolicyID", "tripID", "avg_time_diff_s"]

# Print the five-number summary of the trip-wise sampling frequency.
print(trip_sampling["avg_time_diff_s"].describe())

# Print percentage of trips with avg sampling frequency below 5 seconds.
print(f"\nUnder 5s: {(trip_sampling['avg_time_diff_s'] <= 5).mean():.3%}")

# Print percentage of trips with avg sampling frequency above 60 seconds.
print(f"\nOver 60s: {(trip_sampling['avg_time_diff_s'] > 60).mean():.3%}")

**Observation:** The average time differential between samples is not of one second. Furthermore, we notice that there are no gaps between 5 and 60 seconds. <br>
**Interpretation:** For this to be true, either there are some large intra-trip jumps in recording time, where the tracker malfunctioned/the car was turned off, or the sampling frequency simply is of more than 1 second.<br>
**Implication:** We will have to look at the maximum intra-trip time differential, along with the trimmed mean in order to understand which of these two scenarions wer are facing.            <br>

In [ ]:
# Create the time_diff column.
sampling_data["time_diff"] = sampling_data.groupby(["PolicyID", "tripID"])["timestamp"].diff().dt.total_seconds()

# Sort the column's values ascendingly.
sampling_data["time_diff"].sort_values(ascending = True)

# Compute statistics for intra-trip sampling gaps.
trip_gap_stats = sampling_data.groupby(["PolicyID", "tripID"])["time_diff"].agg(
    avg_diff = "mean",
    max_gap = "max",
    n_gaps_over_5s = lambda x: (x > 5).sum()
).reset_index()

# Print the five-number summary of the statistics.
print(trip_gap_stats[["avg_diff","max_gap", "n_gaps_over_5s"]].describe())

**Observation:** The maximum gap seems to be centered at around 2 seconds, while the average differential is centered at around 1 second. In addition, the number of gaps over 5s is only above 1 for around 25% of trips.<br>
**Interpretation:** Whilst some of these entries have a massive maximum gap, it might be because the tracking system recorded two different trips as a single one, since, for the maximum gap, the time differential is of ~ 3.17 million seconds, or around 36.6 days. <br>
**Implication:** In order to deal with this in a way that maintains the highest amount of data possible, we will nullify the observations which have a time differential of over 5 seconds, as this will allow our data to be as faithful to reality as possible.   <br>

**Final Thoughts:** Some trips have incredibly large time differentials, to address this, we will nullify any data that had a previous gap of over 5 seconds. This will let us maintain the most signal from our data whilst also ensuring that the timestamp column is trustworthy, as this is essential for our future feature engineering.



In [18]:
# Convert timestamp to datetime.
tele_data["timestamp"] = pd.to_datetime(tele_data["timestamp"], format="%d-%m-%Y %H:%M:%S")

# Sort to ensure rows are ordered by PolicyID -> tripID -> timestamp.
tele_data = tele_data.sort_values(["PolicyID", "tripID", "timestamp"]).reset_index(drop = True)

# Calculate the time differential from the past row for each row.
tele_data["time_diff"] = tele_data.groupby(["PolicyID", "tripID"])["timestamp"].diff().dt.total_seconds()

# Create a new column nullifying the values of rows where the time differential is over five seconds.
tele_data["time_diff_clean"] = tele_data["time_diff"].where(tele_data["time_diff"] <= 5, other = np.nan)


## **Feature Diagnostics**

Whilst examining the structure of our data, we noticed that there are some features which require further investigation, including: 
* battery
* coolant_temp_c
* intake_air_temp
* imap
* gps_speed_kph (and it's relationship to speed)
* Accelerometer features (x, y, z)
* Magnetometer features (mx, my, mz)
* trip length and it's distribution
  

Despite the issues with them, we will not be doing further exploration into battery, coolant_temp_c, intake_air_temp, and imap, as these features reflect engine and vehicle state rather than driver behavior. Furthermore, whilst the kmpl and engine_load features are only partially impacted by driver behaviour, they introduce noise, along with being largely collinear with rpm and throttle_pos. Since our goal is behavioral segmentation, they carry no direct signal for our classification task and will be excluded from feature engineering. With this in mind, we just have to explore the accelerometer and magnetometer features, trip length, and gps_speed_kph.

We'll begin with the exploration into the distribution of trip length.

In [ ]:
# Five-number summary of trip duration in minutes.
trip_length_df = tele_data.groupby(["PolicyID", "tripID"]).agg(
    # Calculate real trip duration by summing the time differentials that are smaller than 5 seconds.
    trip_duration_clean = ("time_diff_clean",lambda x: x.sum() / 60)
)
trip_length_df["trip_duration_clean"].describe()

**Observation:** We notice that more than 25% of the trips lasted less than 7 minutes.         <br>
**Interpretation:** Some of these trips will prove to be too small to be statistically relevant, and we will have to find a way to filter them in order to limit the noise in our data.     <br>
**Implication:** We will have to decide on a threshold for trip length in order to determine statistical relevance.         <br>

In [ ]:
# Histogram of trip duration in minutes.
sns.histplot(data = trip_length_df["trip_duration_clean"]).set(title = "Histogram of trip duration in minutes.")

**Observation:** The duration of trips follows a unimodal, heavily right skewed distribution. <br>
**Interpretation:** Due to the fact that nearly 50% of the trips are shorter than 11 minutes, using a threshold to filter and eliminate short trips would mean that we waste a large amount of observations. We will have to develop features which are based upon frequency over time (eg. harsh turns per minute).<br>
**Implication:** We will have engineer features that reduce the importance of exposure, as we care about behaviour, not about time on the road. <br>

### **Exploration of Magnetometer and Accelerometer Data**

In the Data Structure exploration part of this notebook, we discovered that there are some observations in which the features relating to the accelerometer and the magnetometer have missing values, so we will start by looking into those. Furthermore, in the data dictionary, we see that z represents the acceleration in the direction that the car is moving in, x represents the horizontal acceleration 

In [ ]:
accel_magnet_cols = ["x", "y", "z", "mx", "my", "mz"]
tele_data[accel_magnet_cols].describe()

In [ ]:
tele_data[tele_data[accel_magnet_cols].isna().any(axis = 1)]

In [ ]:
tele_data[(tele_data["PolicyID"] == 100) & (tele_data["tripID"] == 1)].describe()

**Observation:** The observations with missing values only occur during one trip. When looking deeper into this trip we see that the values for all of the other features are always zero. <br>
**Interpretation:** This trip are probably for a car sitting idle.    <br>
**Implication:** In order to make the data as representative of reality as possible, we will have to eliminate this trip.<br>

In [ ]:
sns.boxplot(data = tele_data[accel_magnet_cols])

**Observation:** We see that the variance of the magnetometer and accelerometer's recordings for y is very small, whilst the variance for the features regarding x and z is much higher. The fact that these features range from -1 to 1 gives me the impression that they have been already been standardized.<br>
**Interpretation:** The z and x variables will almost definitively have some insight to give us about driver behaviour, such as information about how harsh the driver's cornering is, and how harsh their acceleration/deceleration is.   <br>
**Implication:** We should look to get a better understanding of the x and z features. urthermore, the magnetometer does not give us information directly related to driver behaviour, and simply adds dimensionality to our data, so we will not use mx, my, mz for our segmentation.<br>

We want to engineer features defining harsh acceleration and breaking, and in order to do this in the most statistically relevant way, we will look to decide thresholds based on the distribution of "z" (Since it is the acceleration in the direction the car is moving in.).

In [ ]:
sns.histplot(data = tele_data, x = "z")

In [ ]:
print(f" Values of the 45th and 65th percentile of values for acceleration: ({tele_data['z'].quantile(.45)} , {tele_data['z'].quantile(.65)})")

**Observation:** Over 20% of our observations record an acceleration of 0.           <br>
**Interpretation:** These are most likely recording the car while sitting idle, which does not contribute to our intention of finding a threshold for acceleration. The idea of the threshold is to determine an acceleration level which is very high/very low, so we mostly care about when the car is accelerating. <br>
**Implication:** When finding our acceleration thresholds for harsh acceleration/deceleration, we will have to exclude the observations with a value of 0, as we care about the acceleration while the car is in motion.  <br>

In [ ]:
# Filter to non-zero, non-idle observations
# Use speed > 0 as the idle filter rather than a threshold based upon z, as z = 0 represents constant speed
moving = tele_data[tele_data["speed"] > 0]

# Z thresholds (longitudinal — braking and acceleration)
z_harsh_brake = moving["z"].quantile(0.05)   # bottom 5% = harshest braking
z_harsh_accel = moving["z"].quantile(0.95)   # top 5% = harshest acceleration

# X thresholds (lateral — cornering)
x_harsh_corner = moving["x"].abs().quantile(0.95)  # top 5% magnitude = harshest corners

print(f"Harsh brake threshold (z): {z_harsh_brake:.4f}")
print(f"Harsh accel threshold (z): {z_harsh_accel:.4f}")
print(f"Harsh corner threshold (x): {x_harsh_corner:.4f}")

**Final Thoughts:** From this section, we discovered a trip which was completely idle, along with coming up with some very useful thresholds for harsh braking, acceleration, and cornering. We will need to drop the trips which have a mean speed of 0, in order to ensure that we do not keep any trips where the car is  idle.

In [ ]:
# Make a new dataframe with each trip's mean speed
zero_speed_trips = tele_data.groupby(["PolicyID", "tripID"])["speed"].mean()

# Get a keep only the trips which had a mean speed of 0
zero_speed_trips = zero_speed_trips[zero_speed_trips < 1].reset_index()[["PolicyID","tripID"]]

# Make a set of all of the PolicyID and tripID pairings with mean speed of 0
bad_keys = set(zip(zero_speed_trips["PolicyID"], zero_speed_trips["tripID"]))

tele_data = tele_data[~tele_data.set_index(['PolicyID', 'tripID']).index.isin(bad_keys)].reset_index(drop=True)

print(f"Trips removed: {len(zero_speed_trips)}")
print(f"Rows remaining: {len(tele_data)}")

In [ ]:
tele_data.shape

### **Exploration of Sensor Data / Sensor Sanity**

Here, we will specifically focus on gauging which of the two measures of speed (gps_speed_kph and speed) are more accurate, along with examining their correlation.

In [ ]:
fig, ax = plt.subplots(2,2, figsize = (8,8))

sns.scatterplot(data = tele_data ,
                x = "gps_speed_kph", 
                y = "speed", 
                alpha = 0.6, 
                ax = ax[0,0])

ax[0,0].set_title("Scatterplot of speed to gps_speed_kph")

sns.kdeplot(data = tele_data, 
            x = "speed",
            y = "gps_speed_kph", 
            ax = ax[0,1])
ax[0,1].set_title("Density Plot for speed and gas_speed_kph")


sns.boxplot(data = tele_data["gps_speed_kph"],
            ax = ax[1,0])
ax[1,0].set_title("Boxplot for gps_speed_kph")

sns.boxplot(data = tele_data["speed"],
            ax = ax[1,1])
ax[1,1].set_title("Boxplot for speed")

plt.tight_layout()
plt.show()

**Final Thoughts:** The OBD speed feature is the more reliable of the two speed measures. gps_speed_kph contains physically implausible outliers (>512 km/h) not present in speed, and the two signals are strongly correlated at normal operating values. All downstream feature engineering uses speed exclusively. gps_speed_kph is retained in the dataset but not used.

## **Engineering of Trip-wise Features**

Before we endeavor into feature engineering, it's important that we distinguish driver-behavioral features from engine-state features and focus our feature engineering on the former.

In [59]:
# Distance increment per observation (speed in km/h × 1s = km)
tele_data['distance_increment_km'] = tele_data['speed'] * tele_data['time_diff_clean'] / 3600

# RPM to speed ratio — proxy for aggressive gear usage
tele_data['rpm_speed_ratio'] = tele_data['rpm'] / (tele_data['speed'] + 1)

# Speed-derived acceleration (km/h per second) — more reliable than raw accelerometer
tele_data['speed_delta'] = tele_data.groupby(['PolicyID', 'tripID'])['speed'].diff() / tele_data['time_diff_clean']

# Night driving flag (10pm–5am) — known actuarial risk factor
tele_data['hour'] = tele_data['timestamp'].dt.hour
tele_data['is_night'] = (tele_data['hour'].between(22, 23) | tele_data['hour'].between(0, 5)).astype(int)

# Stop event — transition from moving to stationary, proxy for urban driving
tele_data['is_stopped'] = (tele_data['speed'] == 0).astype(int)
tele_data['stop_event'] = (tele_data.groupby(['PolicyID', 'tripID'])['is_stopped'].diff() == 1).astype(int)

# Create a flag for whether each observation exceeds the threshold
tele_data['is_harsh_brake']  = (tele_data['z'] < z_harsh_brake).astype(int)
tele_data['is_harsh_accel']  = (tele_data['z'] > z_harsh_accel).astype(int)
tele_data['is_harsh_corner'] = (tele_data['x'].abs() > x_harsh_corner).astype(int)

# Detect the START of each event — diff == 1 means transitioned from 0 to 1
tele_data['harsh_brake_event']  = tele_data.groupby(['PolicyID', 'tripID'])['is_harsh_brake'].diff().eq(1).astype(int)
tele_data['harsh_accel_event']  = tele_data.groupby(['PolicyID', 'tripID'])['is_harsh_accel'].diff().eq(1).astype(int)
tele_data['harsh_corner_event'] = tele_data.groupby(['PolicyID', 'tripID'])['is_harsh_corner'].diff().eq(1).astype(int)


# ---Trip-level aggregation
tripwise_data = tele_data.groupby(['PolicyID', 'tripID']).agg(

    trip_duration_min    = ('time_diff_clean', lambda x: x.sum() / 60),
    trip_distance_km     = ('distance_increment_km', 'sum'),

    avg_speed            = ('speed', 'mean'),
    std_speed            = ('speed', 'std'),
    p95_speed            = ('speed', lambda x: x.quantile(0.95)),
    pct_time_over_100    = ('speed', lambda x: (x > 100).mean()),
    pct_time_over_120    = ('speed', lambda x: (x > 120).mean()),
    pct_time_stopped     = ('speed', lambda x: (x == 0).mean()),

    harsh_brakes  = ('harsh_brake_event',  'sum'),
    harsh_accels  = ('harsh_accel_event',  'sum'),
    harsh_corners = ('harsh_corner_event', 'sum'),

    std_accel            = ('speed_delta', 'std'),
    p95_accel            = ('speed_delta', lambda x: x.quantile(0.95)),
    p05_accel            = ('speed_delta', lambda x: x.quantile(0.05)),

    avg_rpm              = ('rpm', 'mean'),
    p95_rpm              = ('rpm', lambda x: x.quantile(0.95)),
    pct_rpm_over_3000    = ('rpm', lambda x: (x > 3000).mean()),
    avg_rpm_speed_ratio  = ('rpm_speed_ratio', 'mean'),

    avg_throttle         = ('throttle_pos', 'mean'),
    std_throttle         = ('throttle_pos', 'std'),
    pct_throttle_over_70 = ('throttle_pos', lambda x: (x > 70).mean()),

    pct_night_driving    = ('is_night', 'mean'),
    n_stop_events        = ('stop_event', 'sum'),

).reset_index()

# Normalize harsh events by distance
tripwise_data['harsh_brakes_per_km']  = tripwise_data['harsh_brakes']  / tripwise_data['trip_distance_km']
tripwise_data['harsh_accels_per_km']  = tripwise_data['harsh_accels']  / tripwise_data['trip_distance_km']
tripwise_data['harsh_corners_per_km'] = tripwise_data['harsh_corners'] / tripwise_data['trip_distance_km']
tripwise_data['stops_per_min']        = tripwise_data['n_stop_events'] / tripwise_data['trip_duration_min']

# Replace inf values (from zero-distance trips) with NaN
tripwise_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"Trips: {len(tripwise_data)}")
tripwise_data.describe(include = "all").T

Trips: 878


,count,mean,std,min,25%,50%,75%,max
PolicyID,878.0,53.232346,30.073013,1.000000,27.000000,53.000000,80.000000,105.000000
tripID,878.0,5.384966,2.870024,1.000000,3.000000,5.000000,8.000000,10.000000
trip_duration_min,878.0,19.156815,21.677478,3.033333,8.504167,12.575000,21.150000,240.483333
trip_distance_km,878.0,7.837583,18.332305,0.046667,1.714444,2.917639,7.189236,213.125556
avg_speed,878.0,16.646791,11.268388,1.027174,9.729854,14.505752,19.327149,79.780159
std_speed,878.0,15.722774,6.319659,2.614845,11.902538,14.994518,18.466552,37.232959
p95_speed,878.0,44.440774,18.225434,7.000000,33.000000,42.000000,52.000000,117.000000
pct_time_over_100,878.0,0.001515,0.016175,0.000000,0.000000,0.000000,0.000000,0.367367
pct_time_over_120,878.0,0.000041,0.000796,0.000000,0.000000,0.000000,0.000000,0.020999
pct_time_stopped,878.0,0.411840,0.183587,0.031320,0.283004,0.388922,0.527823,0.914358


**Observation:** Most features behave as expected, but the per-km harsh driving rates have extreme outliers — some trips record hundreds of harsh events per km. This is almost certainly an artefact of very short or near-zero-distance trips where a handful of harsh events get divided by a negligible denominator.  <br>
**Interpretation:** The inflated per-km rates are a division artefact, not real driving behaviour. They will distort any downstream clustering if not addressed. The 3-minute trip filter we apply next should eliminate most of these cases.  <br>
**Implication:** After filtering, we should verify that the per-km distributions are no longer pathologically skewed, and consider capping extreme outliers before clustering.  <br>

In [ ]:
# Filter out trips under 3 minutes — too short for stable behavioral estimates
MIN_TRIP_DURATION = 3
tripwise_filtered = tripwise_data[tripwise_data['trip_duration_min'] >= MIN_TRIP_DURATION].copy()

print(f"Trips before: {len(tripwise_data)}, after: {len(tripwise_filtered)}")
print(f"Drivers before: {tripwise_data['PolicyID'].nunique()}, after: {tripwise_filtered['PolicyID'].nunique()}")

## **Engineering of Driver-wise Features**

In [ ]:
# Driver-level aggregation 
driver_data = tripwise_filtered.groupby('PolicyID').agg(

    n_trips               = ('tripID', 'count'),
    total_distance_km     = ('trip_distance_km', 'sum'),
    total_duration_min    = ('trip_duration_min', 'sum'),

    avg_speed             = ('avg_speed', 'median'),
    std_speed             = ('std_speed', 'median'),
    p95_speed             = ('p95_speed', 'median'),
    pct_time_over_100     = ('pct_time_over_100', 'median'),
    pct_time_over_120     = ('pct_time_over_120', 'median'),
    pct_time_stopped      = ('pct_time_stopped', 'median'),

    total_harsh_brakes    = ('harsh_brakes', 'sum'),
    total_harsh_accels    = ('harsh_accels', 'sum'),
    total_harsh_corners   = ('harsh_corners', 'sum'),

    std_accel             = ('std_accel', 'median'),
    p95_accel             = ('p95_accel', 'median'),
    p05_accel             = ('p05_accel', 'median'),

    avg_rpm               = ('avg_rpm', 'median'),
    p95_rpm               = ('p95_rpm', 'median'),
    pct_rpm_over_3000     = ('pct_rpm_over_3000', 'median'),
    avg_rpm_speed_ratio   = ('avg_rpm_speed_ratio', 'median'),

    avg_throttle          = ('avg_throttle', 'median'),
    std_throttle          = ('std_throttle', 'median'),
    pct_throttle_over_70  = ('pct_throttle_over_70', 'median'),

    pct_night_driving     = ('pct_night_driving', 'median'),
    stops_per_min         = ('stops_per_min', 'median'),

).reset_index()

# Exposure-normalized harsh event rates at driver level
driver_data['harsh_brakes_per_km']  = driver_data['total_harsh_brakes']  / driver_data['total_distance_km']
driver_data['harsh_accels_per_km']  = driver_data['total_harsh_accels']  / driver_data['total_distance_km']
driver_data['harsh_corners_per_km'] = driver_data['total_harsh_corners'] / driver_data['total_distance_km']

print(f"\nDrivers in driver_data: {len(driver_data)}")
driver_data.describe(include = "all").T

## **Segmentation Prototyping**

#### **K-Means**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features = [
    'harsh_brakes_per_km',
    'harsh_accels_per_km',
    'harsh_corners_per_km',
    'std_speed',
    'p95_speed',
    'avg_rpm_speed_ratio',
    'stops_per_min',
    'pct_time_stopped',
]

# Drop drivers with NaN in any clustering feature
cluster_data = driver_data[['PolicyID'] + cluster_features].dropna()
print(f"Drivers available for clustering: {len(cluster_data)}")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data[cluster_features])

# Elbow plot (inertia) alongside silhouette 
inertias = []
sil_scores_km = {}

for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    score = silhouette_score(X_scaled, labels)
    sil_scores_km[k] = score
    print(f'k={k}: silhouette={score:.4f}  inertia={km.inertia_:.1f}')

best_k_km = max(sil_scores_km, key=sil_scores_km.get)
print(f'\nBest K-Means k={best_k_km} (silhouette={sil_scores_km[best_k_km]:.4f})')

# Fit final K-Means with best k and store labels
km_final = KMeans(n_clusters=best_k_km, random_state=42, n_init=10)
cluster_data['cluster_km'] = km_final.fit_predict(X_scaled)

# Profile each cluster by mean feature values
print('\nK-Means cluster profiles:')
print(cluster_data.groupby('cluster_km')[cluster_features].mean().round(3).T)

#### **K-Medoids (PAM)**

In [ ]:
from sklearn_extra.cluster import KMedoids

sil_scores_kmed = {}

for k in range(2, 7):
    kmed = KMedoids(n_clusters=k, metric='euclidean', random_state=42)
    labels = kmed.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores_kmed[k] = score
    print(f'k={k}: silhouette={score:.4f}')

best_k_kmed = max(sil_scores_kmed, key=sil_scores_kmed.get)
print(f'\nBest K-Medoids k={best_k_kmed} (silhouette={sil_scores_kmed[best_k_kmed]:.4f})')

kmed_final = KMedoids(n_clusters=best_k_kmed, metric='euclidean', random_state=42)
cluster_data['cluster_kmed'] = kmed_final.fit_predict(X_scaled)

print('\nK-Medoids cluster profiles:')
print(cluster_data.groupby('cluster_kmed')[cluster_features].mean().round(3).T)

**Observation:** K-Medoids uses actual data points as cluster centres (medoids) rather than means, making it more robust to outliers. We compare its silhouette score to K-Means to decide which to use.  <br>
**Interpretation:** If the silhouette scores are similar, K-Medoids is preferable given the skewed distributions of our per-km features; if K-Means is substantially higher, the outliers aren't distorting the centroids enough to matter.  <br>
**Implication:** We will select whichever of K-Means or K-Medoids achieves the higher silhouette score and use that method's labels as our final classification variable.  <br>

In [ ]:
# Silhouette score comparison plot — K-Means vs K-Medoids across k
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: silhouette scores
ks = list(range(2, 7))
axes[0].plot(ks, [sil_scores_km[k] for k in ks], marker='o', label='K-Means', color='steelblue')
axes[0].plot(ks, [sil_scores_kmed[k] for k in ks], marker='s', label='K-Medoids', color='coral')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score by k: K-Means vs K-Medoids')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: elbow plot (inertia, K-Means only)
axes[1].plot(ks, inertias, marker='o', color='steelblue')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Inertia')
axes[1].set_title('K-Means Elbow Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observation:** The silhouette scores peak at k=3 for K-Means. We note that K-Medoids performs significantly worse than K-Means for every k. The elbow plot corroborates k=3 as the point of diminishing returns on inertia reduction. <br>
**Interpretation:** Three clusters is the right granularity for this dataset, coarser than two (loses signal) and finer segmentation is not supported by the data's natural structure. <br>
**Implication:** K-Means with k=3 is selected as the final method.

#### **Model Selection & Final Classification**

In [ ]:
# Compare best silhouette scores across methods
best_km   = max(sil_scores_km.values())
best_kmed = max(sil_scores_kmed.values())

print(f'Best K-Means silhouette   : {best_km:.4f}  (k={max(sil_scores_km, key=sil_scores_km.get)})')
print(f'Best K-Medoids silhouette : {best_kmed:.4f}  (k={max(sil_scores_kmed, key=sil_scores_kmed.get)})')

if best_km >= best_kmed:
    print('\n→ Selecting K-Means')
    cluster_data['cluster_raw'] = cluster_data['cluster_km']
    best_method = 'K-Means'
    best_k_final = best_k_km
else:
    print('\n- Selecting K-Medoids')
    cluster_data['cluster_raw'] = cluster_data['cluster_kmed']
    best_method = 'K-Medoids'
    best_k_final = best_k_kmed

#  Risk-rank the clusters by composite risk score 
# Risk score = mean of standardised harsh_brakes_per_km + harsh_accels_per_km
#              + harsh_corners_per_km + p95_speed (all positively correlated with risk)
risk_cols = ['harsh_brakes_per_km', 'harsh_accels_per_km', 'harsh_corners_per_km', 'p95_speed']
profile = cluster_data.groupby('cluster_raw')[risk_cols].mean()
profile['risk_score'] = profile[risk_cols].apply(
    lambda col: (col - col.min()) / (col.max() - col.min())
).mean(axis=1)
profile = profile.sort_values('risk_score')
print('\nCluster risk profiles (sorted low-high risk):')
print(profile.round(4))

# Map original cluster labels to ordered risk labels
label_map = {orig: f'Risk_{i+1}' for i, orig in enumerate(profile.index)}
cluster_data['driver_risk'] = cluster_data['cluster_raw'].map(label_map)
print('\nFinal label distribution:')
print(cluster_data['driver_risk'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

#  A: Cluster profile heatmap 
profile_full = cluster_data.groupby('driver_risk')[cluster_features].mean()

# Normalise each feature 0-1 for heatmap readability
profile_norm = (profile_full - profile_full.min()) / (profile_full.max() - profile_full.min())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Heatmap
im = axes[0].imshow(profile_norm.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks(range(len(cluster_features)))
axes[0].set_xticklabels([f.replace('_', '\n') for f in cluster_features], fontsize=8)
axes[0].set_yticks(range(len(profile_norm)))
axes[0].set_yticklabels(profile_norm.index)
axes[0].set_title('Cluster Profile Heatmap\n(normalised, red = higher risk)')
plt.colorbar(im, ax=axes[0])

#  B: PCA scatter 
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
risk_colors = {'Risk_1': '#2ecc71', 'Risk_2': '#f39c12', 'Risk_3': '#e74c3c'}
for label, grp in cluster_data.groupby('driver_risk'):
    idx = grp.index
    axes[1].scatter(coords[cluster_data.index.get_indexer(idx), 0],
                    coords[cluster_data.index.get_indexer(idx), 1],
                    label=label, color=risk_colors.get(label, 'grey'),
                    alpha=0.8, edgecolors='white', linewidths=0.5, s=80)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('PCA Projection of Driver Clusters')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

#  C: Driver count per risk tier 
counts = cluster_data['driver_risk'].value_counts().sort_index()
bar_colors = [risk_colors.get(r, 'grey') for r in counts.index]
axes[2].bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=0.8)
for i, (label, val) in enumerate(counts.items()):
    axes[2].text(i, val + 0.3, str(val), ha='center', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Risk Tier')
axes[2].set_ylabel('Number of Drivers')
axes[2].set_title('Driver Count by Risk Tier')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nCluster profile (raw feature means):')
print(profile_full.round(4).T)

**Observation:** The three risk tiers are well-separated in PCA space and show distinct profiles in the heatmap. Risk_3 drivers exhibit materially higher per-km harsh event rates and elevated 95th-percentile speeds. Risk_1 drivers show consistently low harsh-event rates and moderate speeds. <br>
**Interpretation:** The clustering has identified a genuine behavioural gradient rather than arbitrary partitions — the separation is visible in raw feature space, not an artefact of the scaling or algorithm. <br>
**Implication:** The driver_risk classification is a meaningful actuarial signal and a credible predictor for the loss cost model.

**Observation:** We rank clusters by a composite risk score derived from the four most actuarially relevant features (per-km harsh events and 95th-percentile speed), mapping the arbitrary cluster integers to interpretable Risk_1 (safest) through Risk_k (riskiest).  <br>
**Interpretation:** This ordering is consistent with insurance underwriting logic, higher harsh-event rates and higher speeds correspond to elevated claim probability.  <br>
**Implication:** The driver_risk column is our final classification variable to be appended to the loss cost data.  <br>

## **Loss Cost Model**

We now merge our driver_risk classification into LC_train.csv and LC_test.csv, then fit the provided Gamma GLM (log link) with driver_risk added as an additional categorical factor. Drivers in the test set whose PolicyID has no telematics data are assigned the modal risk class.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Load LC data 
lc_train = pd.read_csv('../data/LC_train.csv')
lc_test  = pd.read_csv('../data/LC_test.csv')

# Build PolicyID → driver_risk lookup 
risk_lookup = cluster_data[['PolicyID', 'driver_risk']].copy()

# Modal risk class (fallback for unmatched PolicyIDs)
modal_risk = cluster_data['driver_risk'].mode()[0]
print(f'Modal risk class (fallback): {modal_risk}')

# Merge into train and test
lc_train = lc_train.merge(risk_lookup, on='PolicyID', how='left')
lc_train['driver_risk'] = lc_train['driver_risk'].fillna(modal_risk)

lc_test = lc_test.merge(risk_lookup, on='PolicyID', how='left')
lc_test['driver_risk'] = lc_test['driver_risk'].fillna(modal_risk)

print(f'Train shape: {lc_train.shape}  |  Test shape: {lc_test.shape}')
print('\ndriver_risk distribution in train:')
print(lc_train['driver_risk'].value_counts().sort_index())

In [ ]:
# Cast categoricals (mirrors the R as.factor() calls) 
cat_cols = ['pol_pay_freq', 'pol_payd', 'pol_usage',
            'drv_sex1', 'vh_fuel', 'vh_type', 'driver_risk']

for col in cat_cols:
    lc_train[col] = lc_train[col].astype('category')
    lc_test[col]  = lc_test[col].astype('category')

# Align test categories to train (avoids unseen-level errors)
for col in cat_cols:
    lc_test[col] = pd.Categorical(lc_test[col], categories=lc_train[col].cat.categories)

# Fit Gamma GLM (log link) — exact model from the brief + driver_risk 
formula = (
    'claim_amount ~ pol_no_claims_discount + pol_duration + pol_pay_freq + '
    'pol_payd + pol_usage + drv_sex1 + drv_age1 + drv_age_lic2 + '
    'vh_fuel + vh_type + vh_speed + vh_value + vh_weight + driver_risk'
)

glm_fit = smf.glm(
    formula=formula,
    data=lc_train,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

print(glm_fit.summary())

**Observation:** The driver_risk coefficients in the GLM summary show a monotonic increase from Risk_1 (baseline) through Risk_3, consistent with the expected direction — higher-risk drivers produce higher predicted loss costs. <br>
**Interpretation:** The Gamma GLM with log link models the multiplicative relationship between covariates and expected claim amount. A positive coefficient on driver_risk[T.Risk_3] means Risk_3 drivers are predicted to have exp(coef) times the loss cost of the Risk_1 baseline, holding all other policy and vehicle features constant. This is the key result — telematics-derived behaviour improves the model's ability to differentiate risk beyond traditional underwriting variables alone. <br>
**Implication:** Incorporating driver risk segmentation as a rating factor in a PAYD (Pay As You Drive) product would allow the insurer to price more accurately, charging higher premiums to Risk_3 drivers and offering discounts to Risk_1 drivers.

In [ ]:
#  Generate predictions on test set 
lc_test['claim_amount_PREDICTION'] = glm_fit.predict(lc_test)

# Build submission file
submission = lc_test[['PolicyID', 'claim_amount_PREDICTION']].copy()
submission.to_csv('../data/submission.csv', index=False)  # relative to notebook location

print('Submission preview:')
print(submission.head(10))
print(f'\nTotal predictions: {len(submission)}')
print(f'Prediction range: [{submission["claim_amount_PREDICTION"].min():.2f}, '
      f'{submission["claim_amount_PREDICTION"].max():.2f}]')
print('\nSaved to ../data/submission.csv')

## **Conclusion and Observations**

### Summary of Pipeline

1. **Data Cleaning:** Removed idle trip (PolicyID 100, tripID 1), dropped zero-mean-speed trips, nullified time gaps >5s, and dropped uninformative columns (dtc, Unnamed: 0).

2. **Feature Engineering:** Constructed trip-level behavioural features (per-km harsh events, speed percentiles, RPM ratios, night driving, stop frequency), then aggregated to driver-level using medians to reduce trip-length exposure bias.

3. **Clustering:** Evaluated K-Means and K-Medoids for k=2–6. Selected the method and k maximising silhouette score. Clusters were risk-ranked using a composite score of harsh-event rates and 95th-percentile speed.

4. **Loss Cost Model:** Merged driver_risk classification into LC train/test data. Fit a Gamma GLM with log link as specified, with driver_risk as an additional categorical factor. Predictions generated on test set and exported to submission.csv.

### Key Design Decisions

- Used **per-km normalisation** for harsh events rather than raw counts to remove exposure bias from trip length variation.
- Used **driver-level medians** (not means) across trips to reduce sensitivity to outlier trips.
- Excluded engine-state features (coolant_temp, imap, battery, kmpl) from clustering — they reflect vehicle condition, not driver behaviour.
- Magnetometer dropped (no direct behavioural signal); accelerometer z and x retained for longitudinal and lateral dynamics.